# визуализируем значимость для боевого примера

In [1]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import csv
import plotly.figure_factory as ff

In [2]:
from src.schemas import (
    Stage2Out,
    TA_logprob_list,
    TA_tokens_list,
    TA_words_list,
    WordImportance
)

In [3]:
def read_df(path: str):
    df = pd.read_csv(path, quoting=csv.QUOTE_NONNUMERIC, index_col=0)
    df.index = df.index.astype(int)
    return df

- in_perturb_0 - first - perturb5
- in_perturb_1 - mean - perturb5
- in_perturb_2 - first - perturb10
- in_perturb_3 - mean - perturb10

In [6]:
in_checks = "../data/samples_50/modify_prompt_order/checks.csv"
imp_checks = "../data/samples_50/modify_prompt_order/metrics2/importances.csv"
in_perturb_3 = "../data/samples_50/modify_prompt_order/metrics2/perturb1/generate/ptb_results.csv"

base = read_df(in_checks)
importances = read_df(imp_checks)
perturb_3 = read_df(in_perturb_3)

In [7]:
# --- 1. Подготовка данных ---
# Замените эти списки на ваши реальные данные
# Для примера сгенерируем две группы (разного размера, но похожие по природе)
list1 = base["similarity"].tolist()
list2_3 = perturb_3["similarity"].tolist()

In [8]:
(base["similarity"] - perturb_3["similarity"]).sort_values()

check_id
10267   -0.278254
2728    -0.136722
9574    -0.136043
7570    -0.123694
3432    -0.116604
           ...   
21       0.123064
9877     0.137715
11826    0.146581
5952     0.153024
10794    0.175604
Name: similarity, Length: 549, dtype: float64

In [10]:
base.loc[10267]

similarity                                             0.421393
answer                                            О дочери Рут.
passage_id                                                409.0
question      Неужели она хочет, чтобы её сыновья в будущем ...
Name: 10267, dtype: object

In [10]:
base.loc[5992]["answer"]

'О том, что Леди Гага станет первой в истории исполнительницей, выступившей в космосе.'

In [11]:
perturb_3.loc[5992]

similarity                                             0.346347
answer        О том, что Леди Гага станет первой в истории и...
passage_id                                                209.0
question            О чем сообщило новостное издание US Weekly?
passage       (1) Леди Гага станет последней в истории испол...
ptb_words     [{"word": "последней", "start": 21, "end": 30,...
Name: 5992, dtype: object

In [11]:
perturb_3.loc[5992]

similarity                                             0.346347
answer        О том, что Леди Гага станет первой в истории и...
passage_id                                                209.0
question            О чем сообщило новостное издание US Weekly?
passage       (1) Леди Гага станет последней в истории испол...
ptb_words     [{"word": "последней", "start": 21, "end": 30,...
Name: 5992, dtype: object

In [11]:
perturb_3.loc[10267]["ptb_words"]

'[{"word": "также", "start": 13, "end": 18, "importance": -1.0}, {"word": "сообщение", "start": 153, "end": 162, "importance": -1.0}, {"word": "ни", "start": 364, "end": 366, "importance": -1.0}, {"word": "в меру", "start": 581, "end": 587, "importance": -1.0}, {"word": "всё-таки", "start": 1112, "end": 1120, "importance": -1.0}]'

In [14]:
words: list[WordImportance] = TA_words_list.validate_json(importances.loc[10267]["words_importances"])

In [15]:
def create_highlighted_text_html(data: list[WordImportance]) -> str:
    """
    Генерирует HTML-блок с текстом.
    Токены с высокой энтропией подсвечиваются.
    """
    entropy_threshold = 2.1
    max_entropy_scale = 2.5
    html_parts = [
        '<div style="font-family: sans-serif; line-height: 1.6; border: 1px solid #ddd; padding: 15px; border-radius: 8px; background: #f9f9f9;">'
    ]
    for item in data:
        word = item["word"]
        entropy = item["importance"]

        # Логика цвета:
        # Если энтропия < порога -> прозрачный фон.
        # Если выше -> от светло-красного до ярко-красного.
        if entropy > entropy_threshold:
            # Нормализуем альфа-канал от 0.2 до 0.8 в зависимости от силы энтропии
            alpha = min(
                0.8,
                max(
                    0.2,
                    (entropy - entropy_threshold)
                    / (max_entropy_scale - entropy_threshold),
                ),
            )
            bg_color = f"rgba(255, 0, 0, {alpha:.2f})"
            border = "1px solid rgba(255,0,0,0.3)"
        else:
            bg_color = "transparent"
            border = "none"

        # Заменяем переносы строк на <br>, чтобы HTML не ломался
        display_token = word.replace("\n", "<br>")

        # HTML span с тултипом
        span = f"""
<span class="word-span" 
      style="background-color: {bg_color}; border-bottom: {border}; cursor: help; padding: 0 2px; border-radius: 3px;"
      title="Word: '{word}'&#10;Entropy: {entropy:.3f} bits">
{display_token}
</span>
        """
        html_parts.append(span)

    html_parts.append("</div>")
    return "".join(html_parts)


In [16]:
full_html = """
<html>
<head>
    <title>LLM Entropy Analysis</title>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; max-width: 1000px; margin: 0 auto; padding: 20px; color: #333; }
        h2 { border-bottom: 2px solid #333; padding-bottom: 10px; margin-top: 40px; }
        .section { margin-bottom: 50px; }
        .token-span:hover { outline: 2px solid #333; z-index: 10; position: relative; }
    </style>
</head>
<body>
<h1>Анализ "Точек перегиба" (Inflection Points)</h1>
<p>Красным цветом выделены слова, где модель испытывала высокую неуверенность (Internal Confusion).</p>
"""

text_html = create_highlighted_text_html(words)

full_html += f"""
<div class="section">
    <h3>Визуализация текста (наведите на слова):</h3>
    {text_html}
</div>
"""

full_html += "</body></html>"

In [17]:
with open("../reports/Qwen3-32B-v2.html", "w", encoding="utf-8") as f:
    _ = f.write(full_html)